# 📝 API Development (FastAPI)
### Exercises & Solutions — 26 Problems

This notebook is exercises-and-solutions only. It assumes you've already covered the
concept notebook for this topic. Each problem targets a **distinct function, pattern,
or real-world scenario** so that working through all of them gives you practical
exposure to everything commonly used on the job.

**Coverage map:**

- Routes & params: path, query, body, headers, status codes (1-7)
- Pydantic models: validation, nested, validators, response_model (8-13)
- Dependency injection patterns (14-17)
- Error handling & custom exceptions (18-20)
- Middleware, CORS, async endpoints (21-23)
- Testing patterns with TestClient and dependency overrides (24-26)

**Setup:** `pip install fastapi python-multipart` (the latter is needed for the
file-upload exercise, #25).


---


### 1. Path Parameters with Type Conversion

Build a `GET /users/{user_id}` endpoint where `user_id: int` is auto-validated, and test both valid and invalid (non-numeric) inputs.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()

@app.get("/users/{user_id}")
def get_user(user_id: int):
    return {"user_id": user_id, "type": str(type(user_id))}

client = TestClient(app)
print(client.get("/users/42").json())
bad = client.get("/users/not-a-number")
print(f"Invalid input status: {bad.status_code}")

### 2. Multiple Query Parameters with Defaults and Validation

Build a search endpoint `GET /search?q=&limit=&offset=` with sensible defaults and `Query()` constraints (limit capped, offset non-negative).

In [ ]:
from fastapi import FastAPI, Query
from fastapi.testclient import TestClient

app2 = FastAPI()

@app2.get("/search")
def search(q: str = "", limit: int = Query(10, le=100, ge=1), offset: int = Query(0, ge=0)):
    return {"query": q, "limit": limit, "offset": offset}

client2 = TestClient(app2)
print(client2.get("/search?q=python&limit=5").json())
print(client2.get("/search").json())
bad = client2.get("/search?limit=500")
print(f"Over-limit status: {bad.status_code}")

### 3. Enum Path Parameter for Restricted Values

Build `GET /status/{status}` where `status` must be one of a fixed `Enum` set — invalid values automatically rejected with 422.

In [ ]:
from enum import Enum
from fastapi import FastAPI
from fastapi.testclient import TestClient

class OrderStatus(str, Enum):
    pending = "pending"
    shipped = "shipped"
    delivered = "delivered"

app3 = FastAPI()

@app3.get("/status/{status}")
def get_status_info(status: OrderStatus):
    return {"status": status.value, "is_final": status == OrderStatus.delivered}

client3 = TestClient(app3)
print(client3.get("/status/shipped").json())
bad = client3.get("/status/invalid_status")
print(f"Invalid enum value status: {bad.status_code}")

### 4. Request Body with Pydantic Model

Build `POST /items` accepting a Pydantic `Item` model as the JSON body, with required and optional fields.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import Optional

app4 = FastAPI()

class Item(BaseModel):
    name: str
    price: float
    description: Optional[str] = None

@app4.post("/items")
def create_item(item: Item):
    return item.model_dump()

client4 = TestClient(app4)
print(client4.post("/items", json={"name": "Widget", "price": 9.99}).json())
missing = client4.post("/items", json={"name": "Widget"})
print(f"Missing required field status: {missing.status_code}")

### 5. Combining Path, Query, and Body Parameters

Build `PUT /items/{item_id}?notify=` accepting a path param, a query flag, AND a body model all in one endpoint signature.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

app5 = FastAPI()

class ItemUpdate(BaseModel):
    name: str
    price: float

@app5.put("/items/{item_id}")
def update_item(item_id: int, item: ItemUpdate, notify: bool = False):
    return {"item_id": item_id, "updated": item.model_dump(), "notify": notify}

client5 = TestClient(app5)
resp = client5.put("/items/5?notify=true", json={"name": "Updated", "price": 19.99})
print(resp.json())

### 6. Custom Headers as Parameters

Build an endpoint reading a custom `X-Client-Version` header, with a default if absent.

In [ ]:
from fastapi import FastAPI, Header
from fastapi.testclient import TestClient
from typing import Optional

app6 = FastAPI()

@app6.get("/info")
def get_info(x_client_version: Optional[str] = Header(default="unknown")):
    return {"client_version": x_client_version}

client6 = TestClient(app6)
print(client6.get("/info", headers={"X-Client-Version": "2.0.1"}).json())
print(client6.get("/info").json())

### 7. Correct Status Codes per HTTP Method (REST Conventions)

Build a full mini-CRUD with PROPER status codes: 201 for POST, 200 for GET/PUT, 204 for DELETE.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app7 = FastAPI()
db = {}

@app7.post("/widgets", status_code=201)
def create(name: str):
    db[len(db)+1] = name
    return {"id": len(db), "name": name}

@app7.get("/widgets/{id}")
def read(id: int):
    return {"id": id, "name": db.get(id)}

@app7.delete("/widgets/{id}", status_code=204)
def delete(id: int):
    db.pop(id, None)

client7 = TestClient(app7)
created = client7.post("/widgets?name=Gadget")
print(f"POST status: {created.status_code}")
print(f"GET status: {client7.get('/widgets/1').status_code}")
print(f"DELETE status: {client7.delete('/widgets/1').status_code}")

### 8. Nested Pydantic Models

Build an `Order` model containing a list of nested `LineItem` models, and an endpoint computing the total from nested data.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import List

app8 = FastAPI()

class LineItem(BaseModel):
    sku: str
    quantity: int
    unit_price: float

class Order(BaseModel):
    customer: str
    items: List[LineItem]

@app8.post("/orders")
def create_order(order: Order):
    total = sum(i.quantity * i.unit_price for i in order.items)
    return {"customer": order.customer, "item_count": len(order.items), "total": round(total, 2)}

client8 = TestClient(app8)
resp = client8.post("/orders", json={
    "customer": "Alice",
    "items": [{"sku": "A1", "quantity": 2, "unit_price": 10.0}, {"sku": "B2", "quantity": 1, "unit_price": 25.0}]
})
print(resp.json())

### 9. Field Constraints with pydantic.Field

Build a `SignupRequest` model with `Field()` constraints: username length bounds, password minimum length, age range.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

app9 = FastAPI()

class SignupRequest(BaseModel):
    username: str = Field(..., min_length=3, max_length=20)
    password: str = Field(..., min_length=8)
    age: int = Field(..., ge=13, le=120)

@app9.post("/signup")
def signup(req: SignupRequest):
    return {"username": req.username, "status": "created"}

client9 = TestClient(app9)
print(client9.post("/signup", json={"username": "alice", "password": "secret123", "age": 30}).json())
bad = client9.post("/signup", json={"username": "al", "password": "short", "age": 200})
print(f"Multiple violations status: {bad.status_code}")
print(f"Number of errors: {len(bad.json()['detail'])}")

### 10. Custom field_validator for Cross-Field Logic

Build a `DateRange` model with a validator ensuring `end_date` is after `start_date` — a cross-field validation rule.

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, model_validator
from datetime import date

app10 = FastAPI()

class DateRange(BaseModel):
    start_date: date
    end_date: date

    @model_validator(mode="after")
    def check_dates(self):
        if self.end_date <= self.start_date:
            raise ValueError("end_date must be after start_date")
        return self

@app10.post("/bookings")
def create_booking(range_: DateRange):
    return {"nights": (range_.end_date - range_.start_date).days}

client10 = TestClient(app10)
print(client10.post("/bookings", json={"start_date": "2024-01-01", "end_date": "2024-01-05"}).json())
bad = client10.post("/bookings", json={"start_date": "2024-01-05", "end_date": "2024-01-01"})
print(f"Invalid range status: {bad.status_code}")

### 11. response_model to Filter Output Fields

Build a `UserInDB` model (has `password_hash`) and a `UserPublic` response model WITHOUT it, proving `response_model` strips sensitive fields automatically.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

app11 = FastAPI()

class UserPublic(BaseModel):
    id: int
    name: str

@app11.get("/users/{user_id}", response_model=UserPublic)
def get_user(user_id: int):
    # Internal representation includes a sensitive field, but response_model filters it out
    internal_user = {"id": user_id, "name": "Alice", "password_hash": "supersecret_hash"}
    return internal_user

client11 = TestClient(app11)
resp = client11.get("/users/1").json()
print(resp)
print("password_hash leaked:", "password_hash" in resp)

### 12. Optional Fields with Default None and exclude_unset

Build a PATCH-style partial-update endpoint using `exclude_unset=True` so only EXPLICITLY provided fields get applied, leaving others untouched.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import Optional

app12 = FastAPI()
db = {1: {"name": "Widget", "price": 9.99, "in_stock": True}}

class ItemPatch(BaseModel):
    name: Optional[str] = None
    price: Optional[float] = None
    in_stock: Optional[bool] = None

@app12.patch("/items/{item_id}")
def patch_item(item_id: int, patch: ItemPatch):
    updates = patch.model_dump(exclude_unset=True)   # ONLY fields actually sent in the request
    db[item_id].update(updates)
    return db[item_id]

client12 = TestClient(app12)
resp = client12.patch("/items/1", json={"price": 14.99})   # only updating price
print(resp.json())
print("name unchanged:", db[1]["name"] == "Widget")

### 13. List Response Model + Pagination Metadata Wrapper

Build a `PaginatedResponse[T]`-style generic wrapper model containing a list of items plus pagination metadata.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import List, Generic, TypeVar

app13 = FastAPI()

class Product(BaseModel):
    id: int
    name: str

class PaginatedProducts(BaseModel):
    total: int
    page: int
    items: List[Product]

all_products = [Product(id=i, name=f"Product {i}") for i in range(1, 26)]

@app13.get("/products", response_model=PaginatedProducts)
def list_products(page: int = 1, page_size: int = 10):
    start = (page - 1) * page_size
    return PaginatedProducts(
        total=len(all_products), page=page,
        items=all_products[start:start+page_size]
    )

client13 = TestClient(app13)
resp = client13.get("/products?page=2&page_size=5").json()
print(f"Total: {resp['total']}, Page: {resp['page']}, Items: {[i['id'] for i in resp['items']]}")

### 14. Basic Dependency for Shared Query Params

Build a `CommonParams` dependency function shared across MULTIPLE endpoints, avoiding repetition of the same query param logic.

In [ ]:
from fastapi import FastAPI, Depends
from fastapi.testclient import TestClient

app14 = FastAPI()

def common_params(skip: int = 0, limit: int = 20):
    return {"skip": skip, "limit": limit}

@app14.get("/items")
def list_items(params: dict = Depends(common_params)):
    return {"endpoint": "items", **params}

@app14.get("/users")
def list_users(params: dict = Depends(common_params)):
    return {"endpoint": "users", **params}

client14 = TestClient(app14)
print(client14.get("/items?skip=10&limit=5").json())
print(client14.get("/users").json())

### 15. Class-Based Dependency for Stateful Logic

Build a class-based dependency `Pagination` (callable via `__call__`) bundling validation logic into a reusable, instantiable object.

In [ ]:
from fastapi import FastAPI, Depends, Query
from fastapi.testclient import TestClient

app15 = FastAPI()

class Pagination:
    def __init__(self, default_limit=20, max_limit=100):
        self.default_limit, self.max_limit = default_limit, max_limit
    def __call__(self, page: int = Query(1, ge=1), page_size: int = Query(None)):
        size = page_size or self.default_limit
        size = min(size, self.max_limit)
        return {"page": page, "page_size": size, "offset": (page-1)*size}

standard_pagination = Pagination(default_limit=20, max_limit=100)

@app15.get("/records")
def list_records(p: dict = Depends(standard_pagination)):
    return p

client15 = TestClient(app15)
print(client15.get("/records?page=3").json())
print(client15.get("/records?page=1&page_size=500").json())   # capped at max_limit

### 16. Nested Dependencies (Dependency Chains)

Build a chain: `get_db()` → `get_current_user(db)` → endpoint, proving FastAPI resolves multi-level dependency graphs automatically.

In [ ]:
from fastapi import FastAPI, Depends
from fastapi.testclient import TestClient

app16 = FastAPI()

def get_db():
    return {"users": {1: {"name": "Alice", "role": "admin"}}}

def get_current_user(user_id: int = 1, db: dict = Depends(get_db)):
    return db["users"][user_id]

def require_admin(user: dict = Depends(get_current_user)):
    if user["role"] != "admin":
        raise Exception("not authorized")
    return user

@app16.get("/admin-dashboard")
def dashboard(user: dict = Depends(require_admin)):
    return {"welcome": user["name"]}

client16 = TestClient(app16)
print(client16.get("/admin-dashboard").json())

### 17. Dependency Override for Testing (No Real DB Needed)

Show HOW to override a dependency in tests using `app.dependency_overrides`, swapping a real DB dependency for an in-memory fake.

In [ ]:
from fastapi import FastAPI, Depends
from fastapi.testclient import TestClient

app17 = FastAPI()

def get_real_db():
    raise RuntimeError("Would connect to a real production database!")

@app17.get("/health")
def health(db = Depends(get_real_db)):
    return {"db_status": "connected", "db_type": str(type(db))}

client17 = TestClient(app17)

# Without override, this would crash:
try:
    client17.get("/health")
except Exception:
    print("Without override: would have hit the real (broken) dependency")

# Override with a fake for testing:
app17.dependency_overrides[get_real_db] = lambda: {"fake": "db"}
resp = client17.get("/health")
print("With override:", resp.json())

### 18. Custom Domain Exception with Exception Handler

Build a custom `InsufficientStockError` exception and a global handler converting it to a clean `409 Conflict` JSON response.

In [ ]:
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient

app18 = FastAPI()

class InsufficientStockError(Exception):
    def __init__(self, sku, requested, available):
        self.sku, self.requested, self.available = sku, requested, available

@app18.exception_handler(InsufficientStockError)
async def stock_error_handler(request: Request, exc: InsufficientStockError):
    return JSONResponse(status_code=409, content={
        "error": "insufficient_stock", "sku": exc.sku,
        "requested": exc.requested, "available": exc.available
    })

@app18.post("/orders/{sku}")
def order(sku: str, qty: int):
    available = 5
    if qty > available:
        raise InsufficientStockError(sku, qty, available)
    return {"sku": sku, "ordered": qty}

client18 = TestClient(app18)
resp = client18.post("/orders/WIDGET?qty=10")
print(f"Status: {resp.status_code}, Body: {resp.json()}")

### 19. Validation Error Handler with Friendlier Format

Override FastAPI's default 422 validation error response to a simpler, custom-formatted structure using `RequestValidationError`.

In [ ]:
from fastapi import FastAPI, Request
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel

app19 = FastAPI()

@app19.exception_handler(RequestValidationError)
async def custom_validation_handler(request: Request, exc: RequestValidationError):
    simple_errors = [{"field": ".".join(str(p) for p in e["loc"][1:]), "issue": e["msg"]} for e in exc.errors()]
    return JSONResponse(status_code=422, content={"validation_errors": simple_errors})

class Item(BaseModel):
    name: str
    price: float

@app19.post("/items")
def create(item: Item):
    return item

client19 = TestClient(app19)
resp = client19.post("/items", json={"name": "Widget"})   # missing price
print(resp.json())

### 20. HTTPException with Custom Headers

Build an authentication-failure response that includes a `WWW-Authenticate` header alongside the standard `HTTPException` 401.

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

app20 = FastAPI()

@app20.get("/secure")
def secure_endpoint(token: str = None):
    if token != "valid-token":
        raise HTTPException(
            status_code=401,
            detail="Invalid authentication credentials",
            headers={"WWW-Authenticate": "Bearer"},
        )
    return {"message": "authenticated"}

client20 = TestClient(app20)
resp = client20.get("/secure")
print(f"Status: {resp.status_code}")
print(f"WWW-Authenticate header: {resp.headers.get('www-authenticate')}")
print(client20.get("/secure?token=valid-token").json())

### 21. Custom Middleware for Request Timing + Logging

Build middleware that logs EVERY request's method, path, and processing time, attaching it as a response header.

In [ ]:
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient
import time

app21 = FastAPI()
request_log = []

@app21.middleware("http")
async def log_requests(request: Request, call_next):
    start = time.perf_counter()
    response = await call_next(request)
    duration = time.perf_counter() - start
    request_log.append({"method": request.method, "path": request.url.path, "duration_ms": round(duration*1000, 2)})
    response.headers["X-Process-Time-Ms"] = str(round(duration * 1000, 2))
    return response

@app21.get("/ping")
def ping():
    return {"pong": True}

client21 = TestClient(app21)
resp = client21.get("/ping")
print("Header present:", "x-process-time-ms" in resp.headers)
print("Request log:", request_log)

### 22. CORS Middleware Configuration

Configure `CORSMiddleware` restricting allowed origins, and verify the `Access-Control-Allow-Origin` header appears correctly for an allowed origin.

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.testclient import TestClient

app22 = FastAPI()
app22.add_middleware(
    CORSMiddleware,
    allow_origins=["https://trusted-frontend.com"],
    allow_methods=["GET", "POST"],
    allow_headers=["*"],
)

@app22.get("/data")
def get_data():
    return {"data": "value"}

client22 = TestClient(app22)
resp = client22.get("/data", headers={"Origin": "https://trusted-frontend.com"})
print("CORS header:", resp.headers.get("access-control-allow-origin"))

untrusted = client22.get("/data", headers={"Origin": "https://evil-site.com"})
print("Untrusted origin CORS header:", untrusted.headers.get("access-control-allow-origin"))

### 23. Async Endpoint Calling Multiple Async Operations Concurrently

Build an async endpoint that fans out to MULTIPLE async operations concurrently with `asyncio.gather` inside the route handler itself.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
import asyncio

app23 = FastAPI()

async def fetch_inventory(sku):
    await asyncio.sleep(0.02)
    return {"sku": sku, "stock": 42}

async def fetch_price(sku):
    await asyncio.sleep(0.02)
    return {"sku": sku, "price": 19.99}

@app23.get("/product/{sku}")
async def get_product(sku: str):
    inventory, price = await asyncio.gather(fetch_inventory(sku), fetch_price(sku))
    return {"sku": sku, "stock": inventory["stock"], "price": price["price"]}

client23 = TestClient(app23)
print(client23.get("/product/WIDGET-1").json())

### 24. Testing with TestClient Sessions and Headers Persistence

Use `TestClient` to simulate a multi-request session, setting a default header once that persists across multiple calls.

In [ ]:
from fastapi import FastAPI, Header
from fastapi.testclient import TestClient
from typing import Optional

app24 = FastAPI()

@app24.get("/whoami")
def whoami(authorization: Optional[str] = Header(default=None)):
    return {"authenticated": authorization == "Bearer secret-token"}

client24 = TestClient(app24, headers={"Authorization": "Bearer secret-token"})
print(client24.get("/whoami").json())   # header automatically included on every request
print(client24.get("/whoami").json())   # still authenticated, no need to repeat header

### 25. Testing File Upload Endpoints

Build an endpoint accepting file uploads via `UploadFile`, and test it with `TestClient` using the `files=` parameter.

In [ ]:
from fastapi import FastAPI, UploadFile
from fastapi.testclient import TestClient
import io

app25 = FastAPI()

@app25.post("/upload")
async def upload_file(file: UploadFile):
    content = await file.read()
    return {"filename": file.filename, "size_bytes": len(content), "content_type": file.content_type}

client25 = TestClient(app25)
fake_file = io.BytesIO(b"hello world, this is test content")
resp = client25.post("/upload", files={"file": ("test.txt", fake_file, "text/plain")})
print(resp.json())

### 26. Full Integration Test: Multi-Step Workflow

Write a single test exercising a COMPLETE workflow: create a resource, fetch it, update it, then delete it — verifying state at each step.

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel

app26 = FastAPI()
db = {}
next_id = [1]

class Task(BaseModel):
    title: str
    done: bool = False

@app26.post("/tasks", status_code=201)
def create(task: Task):
    tid = next_id[0]; next_id[0] += 1
    db[tid] = task.model_dump()
    return {"id": tid, **db[tid]}

@app26.get("/tasks/{tid}")
def read(tid: int):
    if tid not in db: raise HTTPException(404)
    return {"id": tid, **db[tid]}

@app26.put("/tasks/{tid}")
def update(tid: int, task: Task):
    if tid not in db: raise HTTPException(404)
    db[tid] = task.model_dump()
    return {"id": tid, **db[tid]}

@app26.delete("/tasks/{tid}", status_code=204)
def delete(tid: int):
    db.pop(tid, None)

client26 = TestClient(app26)

# Full lifecycle test
created = client26.post("/tasks", json={"title": "Write tests"}).json()
print("1. Created:", created)

fetched = client26.get(f"/tasks/{created['id']}").json()
assert fetched["title"] == "Write tests"
print("2. Fetched:", fetched)

updated = client26.put(f"/tasks/{created['id']}", json={"title": "Write tests", "done": True}).json()
assert updated["done"] is True
print("3. Updated:", updated)

del_resp = client26.delete(f"/tasks/{created['id']}")
assert del_resp.status_code == 204
print("4. Deleted, status:", del_resp.status_code)

gone = client26.get(f"/tasks/{created['id']}")
assert gone.status_code == 404
print("5. Confirmed gone:", gone.status_code)